# Graphs

Notes and runnable examples on graphs — vertices joined by edges — the most general structure in this series, and the bridge from data structures into algorithms. Python ships **no graph type**; you build one from a `dict`, then traverse it with tools from the earlier notebooks: a `deque` for BFS, recursion/stack for DFS, and `heapq` for weighted shortest paths.

**Contents**

1. **What it is** — vertices, edges, the key axes
2. **Representations** — adjacency list vs matrix
3. **Traversals** — BFS & DFS (and the cycle guard)
4. **What traversals give you** — shortest paths & topological sort
5. **Weighted shortest paths** — Dijkstra with `heapq`
6. **When to use what**
7. **Complexity cheat-sheet**

## 1. What it is

A **graph** is a set of **vertices** (nodes) connected by **edges**. It's the most general structure here — a *tree* is just a graph with no cycles and a single root, and a *linked list* is a graph where every node has one outgoing edge. Graphs vary along a few axes:

- **Directed vs undirected** — do edges have a direction (one-way streets) or go both ways (friendships)?
- **Weighted vs unweighted** — do edges carry a cost/distance, or merely exist?
- **Cyclic vs acyclic** — a **DAG** (directed acyclic graph) has no cycles and powers dependency resolution, scheduling, and build systems.

Since Python has no built-in graph, the first decision is how to **represent** one.

## 2. Representations — adjacency list vs matrix

Two standard layouts with opposite trade-offs:

- **Adjacency list** — a `dict` mapping each vertex to its neighbors (a `list` or `set`). Space is **O(V + E)** and iterating a vertex's neighbors is cheap. This is the default for the **sparse** graphs you meet in practice (E ≪ V²).
- **Adjacency matrix** — a V×V grid where `M[u][v]` marks an edge. Checking an edge is **O(1)**, but space is **O(V²)** no matter how few edges exist — only worth it for **dense** graphs or lookup-heavy work.

In [1]:
graph = {
    "A": ["B", "C"],
    "B": ["A", "D", "E"],
    "C": ["A", "F"],
    "D": ["B"],
    "E": ["B", "F"],
    "F": ["C", "E"],
}                                       # adjacency list (undirected)

print("neighbors of B:", graph["B"])

# the same graph as an adjacency matrix
nodes = list(graph)
idx = {n: i for i, n in enumerate(nodes)}
M = [[0] * len(nodes) for _ in nodes]
for u in graph:
    for v in graph[u]:
        M[idx[u]][idx[v]] = 1

print("    " + " ".join(nodes))
for n in nodes:
    print(f"  {n} " + " ".join(str(x) for x in M[idx[n]]))
print("edge A-B?", M[idx["A"]][idx["B"]], "| edge A-D?", M[idx["A"]][idx["D"]])


neighbors of B: ['A', 'D', 'E']
    A B C D E F
  A 0 1 1 0 0 0
  B 1 0 0 1 1 0
  C 1 0 0 0 0 1
  D 0 1 0 0 0 0
  E 0 1 0 0 0 1
  F 0 0 1 0 1 0
edge A-B? 1 | edge A-D? 0


## 3. Traversals — BFS & DFS

Visiting every reachable vertex uses the same two strategies as the trees notebook, now generalized — with one crucial addition: graphs can contain **cycles**, so you must track a **`visited` set** or you'll loop forever.

- **BFS** explores in rings outward from the source using a **queue** (`deque`) — nearer vertices first.
- **DFS** plunges down one path before backtracking, via **recursion** (an implicit stack) or an explicit `list` stack.

In [2]:
from collections import deque

def bfs(g, start):
    seen, order, q = {start}, [], deque([start])
    while q:
        u = q.popleft()
        order.append(u)
        for v in g[u]:
            if v not in seen:        # the visited guard stops cycles from looping forever
                seen.add(v)
                q.append(v)
    return order

def dfs(g, start, seen=None, order=None):
    if seen is None:
        seen, order = set(), []
    seen.add(start)
    order.append(start)
    for v in g[start]:
        if v not in seen:
            dfs(g, v, seen, order)
    return order

print("BFS from A:", bfs(graph, "A"))
print("DFS from A:", dfs(graph, "A"))


BFS from A: ['A', 'B', 'C', 'D', 'E', 'F']
DFS from A: ['A', 'B', 'D', 'E', 'F', 'C']


## 4. What traversals give you

The traversals aren't the goal — they're the engine under a pile of classic results:

- **BFS = shortest paths in an *unweighted* graph.** Because BFS reaches vertices in ring order, the first time you see a vertex it's along a path with the fewest edges — just record the distance as you go.
- **Topological sort** orders a **DAG** so every edge points "forward" — a valid order for tasks with dependencies (build steps, course prerequisites). **Kahn's algorithm** uses a queue: repeatedly emit a vertex with **in-degree 0**, then drop its out-edges. If you can't empty the graph, it has a cycle.

In [3]:
def bfs_distances(g, start):
    dist, q = {start: 0}, deque([start])
    while q:
        u = q.popleft()
        for v in g[u]:
            if v not in dist:
                dist[v] = dist[u] + 1     # one more edge than where we came from
                q.append(v)
    return dist

print("unweighted distances from A:", bfs_distances(graph, "A"))

# a DAG of course prerequisites: edge u -> v means "u must come before v"
prereqs = {
    "calc1": ["calc2", "physics"], "calc2": ["calc3"], "calc3": [],
    "physics": ["physics2"], "physics2": [],
    "intro_cs": ["algorithms"], "algorithms": [],
}

def topological_sort(dag):
    indeg = {u: 0 for u in dag}
    for u in dag:
        for v in dag[u]:
            indeg[v] += 1
    q = deque(u for u in dag if indeg[u] == 0)     # start from the "no prereqs" vertices
    order = []
    while q:
        u = q.popleft()
        order.append(u)
        for v in dag[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    if len(order) != len(dag):
        raise ValueError("graph has a cycle - no topological order")
    return order

print("a valid study order  :", topological_sort(prereqs))


unweighted distances from A: {'A': 0, 'B': 1, 'C': 1, 'D': 2, 'E': 2, 'F': 2}
a valid study order  : ['calc1', 'intro_cs', 'calc2', 'physics', 'algorithms', 'calc3', 'physics2']


## 5. Weighted shortest paths — Dijkstra with `heapq`

BFS finds the fewest-*edges* path, but once edges carry **weights** (distances, costs) you need **Dijkstra's algorithm**. It's BFS with a twist: instead of a plain queue, pull the *closest-so-far* vertex next using a **min-heap** (`heapq`, from the heaps notebook). Whenever you discover a cheaper route to a vertex, push it with its new distance.

This is where the whole series clicks together: a `dict` graph, a `dict` of best distances, and a `heapq` priority queue.

In [4]:
import heapq

weighted = {
    "A": [("B", 1), ("C", 4)],
    "B": [("C", 2), ("D", 5)],
    "C": [("D", 1)],
    "D": [],
}                                       # edges as (neighbor, weight)

def dijkstra(g, start):
    dist = {start: 0}
    pq = [(0, start)]                   # (distance_so_far, vertex)
    while pq:
        d, u = heapq.heappop(pq)        # always expand the nearest frontier vertex
        if d > dist.get(u, float("inf")):
            continue                    # stale entry - we already found a shorter way
        for v, w in g[u]:
            nd = d + w
            if nd < dist.get(v, float("inf")):
                dist[v] = nd
                heapq.heappush(pq, (nd, v))
    return dist

print("weighted shortest distances from A:", dijkstra(weighted, "A"))
print("(A->B->C = 3 beats A->C = 4;  A->B->C->D = 4)")


weighted shortest distances from A: {'A': 0, 'B': 1, 'C': 3, 'D': 4}
(A->B->C = 3 beats A->C = 4;  A->B->C->D = 4)


## 6. When to use what

| You need... | Reach for | Why |
|---|---|---|
| Represent a sparse graph (most real ones) | adjacency list (`dict` of lists/sets) | O(V+E) space; fast neighbor iteration |
| Represent a dense graph / O(1) edge tests | adjacency matrix | O(1) lookup, at O(V²) space |
| Visit everything / test connectivity | BFS or DFS | both O(V+E); choose by what you need next |
| Shortest path, unweighted | BFS | ring order ⇒ fewest edges |
| Shortest path, weighted (non-negative) | Dijkstra (`heapq`) | greedily expand the nearest, O((V+E) log V) |
| Order tasks with dependencies | topological sort (Kahn / DFS) | linearizes a DAG; detects cycles |
| Shortest path with negative weights | Bellman-Ford (→ `algorithms/`) | Dijkstra assumes non-negative edges |

## 7. Complexity cheat-sheet

| Operation | Adjacency list | Adjacency matrix |
|---|:---:|:---:|
| Space | O(V + E) | O(V²) |
| Add edge | O(1) | O(1) |
| Has edge u–v? | O(deg u) | O(1) |
| Iterate u's neighbors | O(deg u) | O(V) |
| BFS / DFS (full) | O(V + E) | O(V²) |
| Dijkstra (binary heap) | O((V + E) log V) | O(V² + E log V) |
| Topological sort | O(V + E) | O(V²) |

<sub>V = vertices, E = edges, deg u = number of edges at vertex u. Real-world graphs are usually **sparse** (E ≪ V²), which is why the adjacency list is the default.</sub>

---

**Where this leads.** Graph *algorithms* — Dijkstra, A\*, Bellman-Ford, minimum spanning trees (Prim / Kruskal), connected components, cycle detection — are where this data-structures track hands off to an `algorithms/` folder. Every one of them is built on exactly the pieces in these notebooks: a `dict` graph plus a `deque`, a `heapq`, or a `set`. That's the whole point of the series — the fancy algorithms are just the right data structures, well chosen.